<a href="https://colab.research.google.com/github/RytisBalt/Ma-ininis-mokymasis/blob/Antras-Kontrolinis/TMMA2_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 0) duomenys
import pandas as pd
from sklearn.model_selection import train_test_split

path = "/content/diamonds.csv"
df = pd.read_csv(path)

# kiekybiniai stulpeliai
qCols = ['carat', 'depth', 'table', 'price', 'x', 'y','z']

# kokybiniai stulpeliai
ordCols = ['cut', 'color', 'clarity']

# df.iloc[:5,]
# df.loc[:,['cut', 'color', 'clarity']].describe()

# atrenkame 5 proc., kad skaičiavimai netruktų per ilgai
df = df.groupby('cut', group_keys=False).apply(pd.DataFrame.sample,random_state=0, frac=.05)

X0, X1, y = df.loc[:,qCols].values,df.loc[:,['color','clarity']].values, df['cut'].values

X0_train, X0_test, X1_train, X1_test, y_train, y_test = train_test_split(X0, X1, y, train_size=0.75,stratify=y, random_state=0)
X0.shape,X1.shape

# Duomenų rinkmenoje diamonds.csv pateikiami duomenys apie deimantus, paimti iš R paketo ggplot2. Stulpelių aprašymas:

# price - kaina USD;

# carat - svoris karatais;

# cut - pjovimo kokybė (Fair, Good, Very Good, Premium, Ideal);

# color - spalva (kinta nuo D (geriausia) iki J (blogiausia));

# clarity - skaidrumas (I1 (prasčiausias), SI2, SI1, VS2, VS1, VVS2, VVS1, IF (geriausias));

# x - ilgis mm (0–10.74);

# y - plotis mm (0–58.9);

# z - gylis mm (0–31.8);

# depth - išvestinis dydis = 2 * z / (x + y);

# table - deimanto viršutinės dalies santykinis plotis.

# cut, color, clarity - kokybiniai požymiai, likę - kiekybiniai požymiai.
X0

<ipython-input-27-90eaf94d9f79>:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('cut', group_keys=False).apply(pd.DataFrame.sample,random_state=0, frac=.05)


array([[ 2.  , 65.9 , 60.  , ...,  7.8 ,  7.73,  5.12],
       [ 0.5 , 66.5 , 60.  , ...,  4.93,  4.85,  3.25],
       [ 0.5 , 56.3 , 65.  , ...,  5.24,  5.21,  2.94],
       ...,
       [ 0.56, 59.8 , 56.  , ...,  5.35,  5.38,  3.21],
       [ 1.01, 59.5 , 59.  , ...,  6.54,  6.6 ,  3.91],
       [ 0.39, 62.4 , 56.  , ...,  4.64,  4.68,  2.91]])

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, classification_report

scaler = StandardScaler()
X0_train = scaler.fit_transform(X0_train)
X0_test = scaler.transform(X0_test)

#Šioje vietoje standartizuosime duomenis, jis reikalingas AVK ir logistines regresijos modeliui, nes ten duomenys yra jautresni
#todel gera praktika yra standartizuoti duomenis, tuo tarpu naivaus bajeso nebutinai reiktu, nes
#modelis priatiko dispersija ir vidurki kiekvienai klasei atskirai, todel standartizavimas nera butinas.


print("1. Tikslinis logistinės regresijos modelis")
log_reg = LogisticRegression(penalty='l1', solver='liblinear', max_iter=1000)
param_grid = {'C': [0.01, 0.1, 1, 10, 100, 1000]}

grid_search_ovr = GridSearchCV(log_reg, param_grid, scoring='roc_auc_ovr', cv=5)
grid_search_ovo = GridSearchCV(log_reg, param_grid, scoring='roc_auc_ovo', cv=5)


grid_search_ovr.fit(X0_train, y_train)
grid_search_ovo.fit(X0_train, y_train)

best_log_reg_ovr = grid_search_ovr.best_estimator_
best_log_reg_ovo = grid_search_ovo.best_estimator_


print("Geriausi tikslinės logistinės regresijos modelio parametrai (AUCovr):", grid_search_ovr.best_params_)
print("Geriausias AUCovr mokymo duomenų aibėje:", grid_search_ovr.best_score_)
print("AUCovr testavimo duomenų aibėje:", roc_auc_score(y_test, best_log_reg_ovr.predict_proba(X0_test), multi_class='ovr'))


print("Geriausi tikslinės logistinės regresijos modelio parametrai (AUCovo):", grid_search_ovo.best_params_)
print("Geriausias AUCovo mokymo duomenų aibėje:", grid_search_ovo.best_score_)
print("AUCovo testavimo duomenų aibėje:", roc_auc_score(y_test, best_log_reg_ovo.predict_proba(X0_test), multi_class='ovo'))


print("\nKlasifikavimas testavimo duomenims (AUCovr):")
print(classification_report(y_test, best_log_reg_ovr.predict(X0_test)))

print("\nKlasifikavimas testavimo duomenims(AUCovo):")
print(classification_report(y_test, best_log_reg_ovo.predict(X0_test)))

1. Tikslinis logistinės regresijos modelis
Geriausi tikslinės logistinės regresijos modelio parametrai (AUCovr): {'C': 10}
Geriausias AUCovr mokymo duomenų aibėje: 0.8269158597900267
AUCovr testavimo duomenų aibėje: 0.8644673804442728
Geriausi tikslinės logistinės regresijos modelio parametrai (AUCovo): {'C': 10}
Geriausias AUCovo mokymo duomenų aibėje: 0.8051015427925996
AUCovo testavimo duomenų aibėje: 0.8383830547530492

Klasifikavimas testavimo duomenims (AUCovr):
              precision    recall  f1-score   support

        Fair       0.89      0.40      0.55        20
        Good       0.00      0.00      0.00        61
       Ideal       0.72      0.92      0.81       270
     Premium       0.73      0.73      0.73       173
   Very Good       0.51      0.50      0.50       151

    accuracy                           0.68       675
   macro avg       0.57      0.51      0.52       675
weighted avg       0.62      0.68      0.64       675


Klasifikavimas testavimo duomenims(AU

In [ ]:
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV

print("2. Tikslinis tiesinis AVK" )
linear_svc = SVC(kernel='linear')
calibrated_svc = CalibratedClassifierCV(linear_svc, cv=5)


calibrated_svc.fit(X0_train, y_train)


y_pred = calibrated_svc.predict(X0_test)
y_pred_proba = calibrated_svc.predict_proba(X0_test)


auc_score_test_lnsvm = roc_auc_score(y_test, y_pred_proba, multi_class='ovr')
print("AUC testavimo duomenų aibėje:", auc_score_test_lnsvm)


print("\nKlasifikavimas testavimo duomenims:")
print(classification_report(y_test, y_pred))

2. Tikslinis tiesinis AVK
AUC testavimo duomenų aibėje: 0.821643603570245

Klasifikavimas testavimo duomenims:
              precision    recall  f1-score   support

        Fair       0.73      0.55      0.63        20
        Good       0.00      0.00      0.00        61
       Ideal       0.74      0.95      0.83       270
     Premium       0.60      0.75      0.66       173
   Very Good       0.46      0.29      0.36       151

    accuracy                           0.65       675
   macro avg       0.51      0.51      0.50       675
weighted avg       0.57      0.65      0.60       675



/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
from sklearn.kernel_approximation import RBFSampler
from sklearn.pipeline import Pipeline
print("3. Tikslinis Gauso AVK")

pipeline = Pipeline([
    ('rbf_sampler', RBFSampler(n_components=100, gamma=1.0)),  # Parametrai pagal nutylejima
    ('svc', CalibratedClassifierCV(SVC(kernel='linear'), cv=5))
])


pipeline.fit(X0_train, y_train)


y_pred_proba = pipeline.predict_proba(X0_test)
y_pred = pipeline.predict(X0_test)

auc_score_test_gaus_avk = roc_auc_score(y_test, y_pred_proba, multi_class='ovr')
print("AUC score mokymo duomenims:", auc_score_test_gaus_avk)


print("\nKlasifikavimas mokymo duomenims:")
print(classification_report(y_test, y_pred))

#Manau gamma parametra vertetu optimizuoti, paziurint gardeleje sakykime nuo [0.01 iki 1000] kuris geriausias parametras
#Paskaites trumpai dokumentacija, jis daugiau daro itakos modeliui ir todel jo optimizavimas pagerintu spejimo kokybe, ko mes ir norim

3. Tikslinis Gauso AVK
AUC score mokymo duomenims: 0.8603340777359405

Klasifikavimas mokymo duomenims:
              precision    recall  f1-score   support

        Fair       0.50      0.25      0.33        20
        Good       0.58      0.51      0.54        61
       Ideal       0.76      0.94      0.84       270
     Premium       0.62      0.79      0.69       173
   Very Good       0.52      0.20      0.29       151

    accuracy                           0.68       675
   macro avg       0.60      0.54      0.54       675
weighted avg       0.65      0.68      0.64       675



In [ ]:
from sklearn.naive_bayes import GaussianNB
print("4. Naivaus Bajeso klasifikatorius")
gnb = GaussianNB()


gnb.fit(X0_train, y_train)

y_pred = gnb.predict(X0_test)
y_pred_proba = gnb.predict_proba(X0_test)


auc_score_test_gnb = roc_auc_score(y_test, y_pred_proba, multi_class='ovr')
print("AUC testavimo duomenims:", auc_score_test_gnb)


print("\nKlasifikavimas testavimo duomenims:")
print(classification_report(y_test, y_pred))

4. Naivaus Bajeso klasifikatorius
AUC testavimo duomenims: 0.8144480523138504

Klasifikavimas testavimo duomenims:
              precision    recall  f1-score   support

        Fair       0.41      0.35      0.38        20
        Good       0.36      0.21      0.27        61
       Ideal       0.70      0.92      0.79       270
     Premium       0.57      0.62      0.59       173
   Very Good       0.40      0.21      0.27       151

    accuracy                           0.60       675
   macro avg       0.49      0.46      0.46       675
weighted avg       0.56      0.60      0.56       675



In [ ]:
auc_score_test_log_reg = roc_auc_score(y_test, best_log_reg_ovr.predict_proba(X0_test), multi_class='ovr')

print("Modelių palyginimas (AUCovr on Test Set):")
print(f"1. Tikslinis logistinės regresijos modelis: AUC = {auc_score_test_log_reg}")
print(f"2. Tikslinis tiesinis AVK: AUC = {auc_score_test_lnsvm }")
print(f"Tikslinis Gauso AVK: AUC = {auc_score_test_gaus_avk}")
print(f"Naivaus Bajeso klasifikatorius: AUC = {auc_score_test_gnb}")


models = ["Logistinės regresijos", "Tiesinis AVK",
          "Tiesinis Gauso AVK", "Naivaus Bajeso"]
auc_scores = [auc_score_test_log_reg , auc_score_test_lnsvm, auc_score_test_gaus_avk , auc_score_test_gnb]
best_model_index = auc_scores.index(max(auc_scores))
best_model = models[best_model_index]

print(f"\nGeriausias modelis: {best_model} su AUC = {auc_scores[best_model_index]}")

#Padarius kiekvieno modelio analizę, atsižvelgsim į auc_score, taip pat yra klasifikavimo lentelės kiekvienam modeliui
#kaip kiekviena deimanto kokybė yra klasifikuojama, sprendžiant is auc geriausias yra logistines regresijos modelis, jo AUC didžiausias
#bet jei yra noro spręsti pagal kitas metrikas, galima tai pasižiūreti klasifikavimo lentelėse, bet rinkčiausi logistinės regresijos
#model5

Modelių palyginimas (AUCovr on Test Set):
1. Tikslinis logistinės regresijos modelis: AUC = 0.8644673804442728
2. Tikslinis tiesinis AVK: AUC = 0.821643603570245
Tikslinis Gauso AVK: AUC = 0.8603340777359405
Naivaus Bajeso klasifikatorius: AUC = 0.8144480523138504

Geriausias modelis: Logistinės regresijos su AUC = 0.8644673804442728
